In [1]:
# Cell 1
import os
import time
import json
import torch
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm
from dotenv import load_dotenv

# specific imports for this notebook
from sentence_transformers import SentenceTransformer
import faiss

load_dotenv()

# Setup device
device = "mps" if torch.backends.mps.is_available() else "cpu"
print(f"Using device: {device}")
print(f"PyTorch version: {torch.__version__}")

# Define paths
PROCESSED_DATA_DIR = "../data/processed"
VECTORSTORE_DIR = "../vectorstore"
os.makedirs(VECTORSTORE_DIR, exist_ok=True)

INPUT_FILE = os.path.join(PROCESSED_DATA_DIR, "chunks.json")
INDEX_FILE = os.path.join(VECTORSTORE_DIR, "faiss_index.bin")
METADATA_FILE = os.path.join(VECTORSTORE_DIR, "chunk_metadata.json")

Using device: mps
PyTorch version: 2.11.0


In [2]:
# Cell 2
print(f"Loading chunks from {INPUT_FILE}...")
start_time = time.time()

if not os.path.exists(INPUT_FILE):
    raise FileNotFoundError(f"Could not find {INPUT_FILE}. Did you run Notebook 1?")

with open(INPUT_FILE, "r", encoding="utf-8") as f:
    document_chunks = json.load(f)

# Extract just the text for encoding
texts = [chunk["chunk_text"] for chunk in document_chunks]

elapsed_time = time.time() - start_time
print(f"Loaded {len(texts)} chunks in {elapsed_time:.4f} seconds.")

Loading chunks from ../data/processed/chunks.json...
Loaded 6422 chunks in 0.0179 seconds.


In [3]:
# Cell 3
print(f"Loading SentenceTransformer model to {device}...")
start_time = time.time()

# This will download the model (~90MB) on the first run
encoder = SentenceTransformer("all-MiniLM-L6-v2", device=device)

elapsed_time = time.time() - start_time
print(f"Model loaded in {elapsed_time:.2f} seconds.")

# Quick test to ensure MPS is handling the model correctly
test_emb = encoder.encode("Hello, this is a test.", convert_to_tensor=True)
print(f"Embedding shape: {test_emb.shape}")
print(f"Tensor device: {test_emb.device}")

Loading SentenceTransformer model to mps...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded in 4.12 seconds.
Embedding shape: torch.Size([384])
Tensor device: mps:0


In [4]:
# Cell 4
print(f"Starting to encode {len(texts)} chunks...")
start_time = time.time()

# encode() automatically batches the inputs. 
# batch_size=64 is highly optimized for M4/16GB.
# show_progress_bar=True uses tqdm under the hood.
embeddings = encoder.encode(
    texts, 
    batch_size=64, 
    show_progress_bar=True, 
    convert_to_numpy=True, # FAISS expects numpy arrays
    normalize_embeddings=True # Normalizing helps with cosine similarity / inner product
)

elapsed_time = time.time() - start_time
print(f"Encoding complete! Generated embeddings of shape {embeddings.shape} in {elapsed_time:.2f} seconds.")

Starting to encode 6422 chunks...


Batches:   0%|          | 0/101 [00:00<?, ?it/s]

Encoding complete! Generated embeddings of shape (6422, 384) in 31.60 seconds.


In [5]:
# Updated Cell 5: Batched Migration to ChromaDB
import chromadb

# 1. Initialize Client
client = chromadb.PersistentClient(path="../vectorstore/chroma_db")
collection = client.get_or_create_collection(name="tisd_knowledge_base")

# 2. Define Batch Size (Safe limit is 5000)
BATCH_SIZE = 4000
total_chunks = len(document_chunks)

# 3. Process in batches
print(f"Migrating {total_chunks} chunks to ChromaDB in batches of {BATCH_SIZE}...")

for i in tqdm(range(0, total_chunks, BATCH_SIZE), desc="Batch Indexing"):
    end_idx = min(i + BATCH_SIZE, total_chunks)
    
    # Slice the data
    batch_docs = [c['chunk_text'] for c in document_chunks[i:end_idx]]
    batch_meta = [{
        "class": str(c.get('class_level', 'unknown')), 
        "subject": str(c.get('subject', 'general')),
        "page": int(c.get('page_num', 0)),
        "source": str(c.get('source_type', 'textbook'))
    } for c in document_chunks[i:end_idx]]
    batch_ids = [str(idx) for idx in range(i, end_idx)]
    batch_embeddings = embeddings[i:end_idx].tolist()
    
    # Add to Chroma
    collection.add(
        documents=batch_docs,
        metadatas=batch_meta,
        ids=batch_ids,
        embeddings=batch_embeddings
    )

print(f"Migration complete! {collection.count()} chunks now indexed in ChromaDB.")

Migrating 6422 chunks to ChromaDB in batches of 4000...


Batch Indexing:   0%|          | 0/2 [00:00<?, ?it/s]

Migration complete! 6422 chunks now indexed in ChromaDB.


In [6]:
# Updated Cell 6 (The new search function for ChromaDB)
def search_textbooks(query, top_k=3, class_filter=None):
    print(f"\nStudent asks: '{query}'")
    
    # 1. ChromaDB query (Handles embedding internally if you use query_texts)
    where_clause = {"class": class_filter} if class_filter else None
    
    results = collection.query(
        query_texts=[query],
        n_results=top_k,
        where=where_clause
    )
    
    # 2. Print the results
    for i in range(len(results['documents'][0])):
        doc = results['documents'][0][i]
        meta = results['metadatas'][0][i]
        
        print(f"Result {i+1}")
        print(f"Class: {meta.get('class')} | Subject: {meta.get('subject')}")
        print(f"Text: {doc[:150]}...") # Print first 150 chars
        print("-" * 30)

# Run a test query (Now with class filtering!)
test_query = "What is the sun and why is it hot?"
search_textbooks(test_query, class_filter="4") # Filtering for Grade 4 only!


Student asks: 'What is the sun and why is it hot?'
Result 1
Class: 4 | Subject: Evs
Text: day? Discuss these changes with your friends and make a list. Can you guess the brightest object in the sky? It is the Sun! In fact, it is so bright t...
------------------------------
Result 2
Class: 4 | Subject: Evs
Text: day? Discuss these changes with your friends and make a list. Can you guess the brightest object in the sky? It is the Sun! In fact, it is so bright t...
------------------------------
Result 3
Class: 4 | Subject: Evs
Text: 157Our Sky Draw the position of the Sun and corresponding shadows in the images given below. Morning Noon Evening The Sun appears to move from the Eas...
------------------------------
